In [2]:
import pandas as pd
import numpy as np

# Task 1: Load and inspect the dataset
df = pd.read_csv('22_restaurant_orders.csv')
print(df.head())
print(df.info())

# Task 2: Check for missing values and duplicate Order_ID records
print("\nMissing values:")
print(df.isnull().sum())

duplicates = df['Order_ID'].duplicated().sum()
print("\nDuplicate Order_ID count:", duplicates)
df = df.drop_duplicates(subset=['Order_ID'])

# Task 3: Clean missing Item_Quantity or Item_Price values
# Filling missing quantity with 1 and missing price with the median price
df['Item_Quantity'] = df['Item_Quantity'].fillna(1)
median_price = df['Item_Price'].median()
df['Item_Price'] = df['Item_Price'].fillna(median_price)
print("\nMissing values after cleaning:")
print(df.isnull().sum())

# Task 4: Convert Order_DateTime to datetime and extract hour and weekday
df['Order_DateTime'] = pd.to_datetime(df['Order_DateTime'])
df['Hour'] = df['Order_DateTime'].dt.hour
df['Weekday'] = df['Order_DateTime'].dt.day_name()
print(df[['Order_DateTime', 'Hour', 'Weekday']].head())

# Task 5: Calculate total orders, total items sold, and total revenue
df['Item_Revenue'] = df['Item_Quantity'] * df['Item_Price']

total_orders = df['Order_ID'].nunique()
total_items = df['Item_Quantity'].sum()
total_revenue = df['Item_Revenue'].sum()

print("\nTotal Orders:", total_orders)
print("Total Items Sold:", total_items)
print("Total Revenue:", total_revenue)


   Order_ID Customer_ID  Dish_Name     Category Order_Type  Item_Quantity  \
0       101        C036      Steak  Main Course   Takeaway            4.0   
1       102        C001    Biryani  Main Course   Delivery            3.0   
2       103        C008  Ice Cream      Dessert   Takeaway            4.0   
3       104        C113       Soup      Starter   Delivery            1.0   
4       105        C099      Fries      Starter   Takeaway            1.0   

   Item_Price       Order_DateTime  
0       950.0  2026-03-28 11:29:00  
1       280.0  2026-03-01 15:42:00  
2       110.0  2026-03-26 09:01:00  
3        90.0  2026-03-07 13:48:00  
4       100.0  2026-03-21 20:08:00  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order_ID        1005 non-null   int64  
 1   Customer_ID     1005 non-null   object 
 2   Dish_Name       1005 non

In [3]:
# Task 6: Calculate row-wise Item_Revenue
df['Item_Revenue'] = df['Item_Quantity'] * df['Item_Price']
print("Task 6 - Item Revenue:")
print(df[['Dish_Name', 'Item_Quantity', 'Item_Price', 'Item_Revenue']].head())

# Task 7: Find the 10 most frequently ordered dishes
top_10_dishes = df['Dish_Name'].value_counts().head(10)
print("\nTask 7 - Top 10 Most Frequently Ordered Dishes:")
print(top_10_dishes)

# Task 8: Rank menu categories by total revenue
category_revenue = df.groupby('Category')['Item_Revenue'].sum().sort_values(ascending=False)
print("\nTask 8 - Menu Categories Ranked by Total Revenue:")
print(category_revenue)

# Task 9: Compare average bill value across dine-in, takeaway, and delivery orders
order_bills = df.groupby(['Order_ID', 'Order_Type'])['Item_Revenue'].sum().reset_index()
avg_bill_by_type = order_bills.groupby('Order_Type')['Item_Revenue'].mean().round(2)
print("\nTask 9 - Average Bill Value by Order Type:")
print(avg_bill_by_type)

# Task 10: Identify peak ordering hours and the busiest day of the week
peak_hours = df['Hour'].value_counts().head(5)
busiest_days = df['Weekday'].value_counts()

print("\nTask 10 - Top 5 Peak Ordering Hours:")
print(peak_hours)
print("\nTask 10 - Orders by Day of the Week:")
print(busiest_days)


Task 6 - Item Revenue:
   Dish_Name  Item_Quantity  Item_Price  Item_Revenue
0      Steak            4.0       950.0        3800.0
1    Biryani            3.0       280.0         840.0
2  Ice Cream            4.0       110.0         440.0
3       Soup            1.0        90.0          90.0
4      Fries            1.0       100.0         100.0

Task 7 - Top 10 Most Frequently Ordered Dishes:
Dish_Name
Burger       95
Coffee       95
Noodles      86
Sandwich     85
Pasta        85
Ice Cream    76
Steak        75
Salad        73
Fries        70
Tacos        68
Name: count, dtype: int64

Task 8 - Menu Categories Ranked by Total Revenue:
Category
Main Course    433940.0
Starter         81780.0
Dessert         22770.0
Beverage        18700.0
Name: Item_Revenue, dtype: float64

Task 9 - Average Bill Value by Order Type:
Order_Type
Delivery    548.18
Dine-in     528.58
Takeaway    599.61
Name: Item_Revenue, dtype: float64

Task 10 - Top 5 Peak Ordering Hours:
Hour
8     75
16    70
10    70


In [4]:
# Task 11: Use a list for top dishes and a tuple for fixed meal categories
top_dishes_list = list(df['Dish_Name'].value_counts().head(5).index)
meal_categories = ('Starter', 'Main Course', 'Dessert', 'Beverage')

print("Task 11 - Top Dishes List:", top_dishes_list)
print("Task 11 - Fixed Categories Tuple:", meal_categories)

# Task 12: Use a set for unique customer IDs and write a function to calculate total bill
unique_customers = set(df['Customer_ID'])
print("\nTask 12 - Total Unique Customers:", len(unique_customers))

def calculate_total_bill(order_id):
    order_data = df[df['Order_ID'] == order_id]
    total = order_data['Item_Revenue'].sum()
    return total

print("Total bill for Order 101:", calculate_total_bill(101))

# Task 13: Use NumPy to calculate mean, median, and maximum bill value
bills = df.groupby('Order_ID')['Item_Revenue'].sum().values

mean_bill = round(np.mean(bills), 2)
median_bill = round(np.median(bills), 2)
max_bill = round(np.max(bills), 2)

print("\nTask 13 - Mean Bill:", mean_bill)
print("Task 13 - Median Bill:", median_bill)
print("Task 13 - Max Bill:", max_bill)

# Task 14: Identify unusually large bills using IQR and create order-time categories
q1 = np.percentile(bills, 25)
q3 = np.percentile(bills, 75)
iqr = q3 - q1
upper_limit = q3 + (1.5 * iqr)

order_totals = df.groupby('Order_ID')['Item_Revenue'].sum().reset_index()
large_bills = order_totals[order_totals['Item_Revenue'] > upper_limit]
print("\nTask 14 - Unusually Large Bills Count:", len(large_bills))
print(large_bills.head())

time_categories = []
for h in df['Hour']:
    if 5 <= h < 12:
        time_categories.append('Morning')
    elif 12 <= h < 17:
        time_categories.append('Afternoon')
    elif 17 <= h < 21:
        time_categories.append('Evening')
    else:
        time_categories.append('Night')

df['Time_Category'] = time_categories
print("\nTask 14 - Order Time Categories:")
print(df[['Order_ID', 'Hour', 'Time_Category']].head())


Task 11 - Top Dishes List: ['Burger', 'Coffee', 'Noodles', 'Sandwich', 'Pasta']
Task 11 - Fixed Categories Tuple: ('Starter', 'Main Course', 'Dessert', 'Beverage')

Task 12 - Total Unique Customers: 244
Total bill for Order 101: 3800.0

Task 13 - Mean Bill: 557.19
Task 13 - Median Bill: 390.0
Task 13 - Max Bill: 3800.0

Task 14 - Unusually Large Bills Count: 73
    Order_ID  Item_Revenue
0        101        3800.0
7        108        3800.0
43       144        2850.0
49       150        3800.0
62       163        1200.0

Task 14 - Order Time Categories:
   Order_ID  Hour Time_Category
0       101    11       Morning
1       102    15     Afternoon
2       103     9       Morning
3       104    13     Afternoon
4       105    20       Evening


In [5]:
# Task 15: Create a simple MenuItem class and write 5 menu-planning recommendations
class MenuItem:
    def __init__(self, name, category, price):
        self.name = name
        self.category = category
        self.price = price

    def show_info(self):
        print(f"Dish: {self.name} | Category: {self.category} | Price: {self.price}")

m1 = MenuItem("Pizza", "Main Course", 300)
m2 = MenuItem("Burger", "Main Course", 150)
m3 = MenuItem("Coffee", "Beverage", 80)

for m in (m1, m2, m3):
    m.show_info()

print("\n--- Menu Planning Recommendations ---")
print("1. Keep top-ordered items well-stocked to avoid shortages during rush hours.")
print("2. Focus promotions on Main Course dishes since they generate the highest total revenue.")
print("3. Create combo deals with Starters and Beverages to increase smaller takeaway bills.")
print("4. Schedule extra kitchen staff during Evening and Night hours when orders peak.")
print("5. Offer special discounts on slower weekdays to balance customer traffic across the week.")


Dish: Pizza | Category: Main Course | Price: 300
Dish: Burger | Category: Main Course | Price: 150
Dish: Coffee | Category: Beverage | Price: 80

--- Menu Planning Recommendations ---
1. Keep top-ordered items well-stocked to avoid shortages during rush hours.
2. Focus promotions on Main Course dishes since they generate the highest total revenue.
3. Create combo deals with Starters and Beverages to increase smaller takeaway bills.
4. Schedule extra kitchen staff during Evening and Night hours when orders peak.
5. Offer special discounts on slower weekdays to balance customer traffic across the week.
